In [1]:
!pip install kaggle

In [2]:
import os
import json
import torch
import difflib
from google.colab import files
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import DonutProcessor, VisionEncoderDecoderModel
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW

In [3]:
# 1. Munculkan tombol upload untuk memilih file kaggle.json
print("Silakan upload file kaggle.json:")
files.upload()

# 2. Buat direktori kaggle, pindahkan file, dan atur permission-nya
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API token berhasil dikonfigurasi!")

Silakan upload file kaggle.json:


Saving kaggle.json to kaggle.json
Kaggle API token berhasil dikonfigurasi!


In [4]:
!kaggle datasets download -d urbikn/sroie-datasetv2 --unzip

Dataset URL: https://www.kaggle.com/datasets/urbikn/sroie-datasetv2
License(s): other
100% 834M/834M [00:03<00:00, 251MB/s]



In [5]:
class SROIEDataset(Dataset):
    def __init__(self, dataset_path, processor, max_length=512):
        self.dataset_path = Path(dataset_path)
        self.processor = processor
        self.max_length = max_length

        # Ambil list file gambar
        self.image_filenames = sorted(list((self.dataset_path / "img").glob("*.jpg")))

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        # 1. Load Gambar
        img_path = self.image_filenames[idx]
        image = Image.open(img_path).convert("RGB")

        # 2. Load Label (.txt di folder entities biasanya berisi string JSON)
        label_path = self.dataset_path / "entities" / f"{img_path.stem}.txt"
        with open(label_path, 'r') as f:
            label_data = json.load(f)

        # 3. Format label ke struktur target Donut: <s_receipt><s_company>...
        # Kita bungkus data JSON ke dalam format ground truth string
        ground_truth_str = json.dumps(label_data)
        target_sequence = f"<s_receipt>{ground_truth_str}</s_receipt>"

        # 4. Preprocessing Gambar (Pixel Values)
        pixel_values = self.processor(image, return_tensors="pt").pixel_values

        # 5. Preprocessing Teks (Labels/Tokenization)
        labels = self.processor.tokenizer(
            target_sequence,
            add_special_tokens=False,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids

        return {
            "pixel_values": pixel_values.squeeze(),
            "labels": labels.squeeze(),
            "target_sequence": target_sequence
        }


In [6]:
!nvidia-smi

Tue May 19 08:35:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
# --- Cara Penggunaan ---
model_id = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(model_id)
model = VisionEncoderDecoderModel.from_pretrained(model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tambahkan special tokens baru ke tokenizer dan resize model embedding
new_tokens = ["<s_receipt>", "</s_receipt>"]
processor.image_processor.size = {"height": 720, "width": 1280}
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.to(device)

# 3. Setup Dataset & Dataloader (Pastikan path benar)
dataset = SROIEDataset(dataset_path="/content/SROIE2019/train", processor=processor)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Testing satu batch
batch = next(iter(dataloader))
print(f"Shape Pixel Values: {batch['pixel_values'].shape}")
print(f"Contoh Target: {batch['target_sequence'][0]}")

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Shape Pixel Values: torch.Size([1, 3, 720, 1280])
Contoh Target: <s_receipt>{"company": "ESJAY FUEL ENTERPRISE", "date": "30-06-2018", "address": "LOT PTD 101051 JALAN PERMAS 10/10 81750 MASAI, JOHOR", "total": "50.00"}</s_receipt>


In [12]:
!nvidia-smi

Tue May 19 08:38:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             31W /   70W |     891MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
torch.cuda.empty_cache()
model.gradient_checkpointing_enable()
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(["<s_receipt>"])[0]
model.train()
optimizer = AdamW(model.decoder.parameters(), lr=2e-5) # Opsional: hanya train decoder agar lebih ringan
epochs = 20

# Setup GradScaler untuk Mixed Precision
scaler = torch.amp.GradScaler('cuda')

for epoch in range(epochs):
    print(f"\nMemulai Epoch {epoch + 1}/{epochs}")
    for batch_idx, batch in enumerate(dataloader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # Gunakan autocast (fp16) agar hemat VRAM
        with torch.amp.autocast('cuda', dtype=torch.float16):
          outputs = model(pixel_values=pixel_values, labels=labels)
          loss = outputs.loss

        # Backward & Optimasi menggunakan Scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx} - Loss: {loss.item():.4f}")


Memulai Epoch 1/20


`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...


Batch 0 - Loss: 31.1490
Batch 10 - Loss: 10.6730
Batch 20 - Loss: 8.4510
Batch 30 - Loss: 7.5875
Batch 40 - Loss: 6.7206
Batch 50 - Loss: 5.7646
Batch 60 - Loss: 5.3559
Batch 70 - Loss: 3.4892
Batch 80 - Loss: 2.4160
Batch 90 - Loss: 1.3570
Batch 100 - Loss: 1.3296
Batch 110 - Loss: 0.5759
Batch 120 - Loss: 0.5528
Batch 130 - Loss: 0.5093
Batch 140 - Loss: 0.8531
Batch 150 - Loss: 0.6175
Batch 160 - Loss: 0.3599
Batch 170 - Loss: 0.3882
Batch 180 - Loss: 0.2917
Batch 190 - Loss: 0.3556
Batch 200 - Loss: 0.2625
Batch 210 - Loss: 0.4266
Batch 220 - Loss: 0.3105
Batch 230 - Loss: 0.3287
Batch 240 - Loss: 0.4350
Batch 250 - Loss: 0.2774
Batch 260 - Loss: 0.5732
Batch 270 - Loss: 0.2254
Batch 280 - Loss: 0.5789
Batch 290 - Loss: 0.4165
Batch 300 - Loss: 0.3272
Batch 310 - Loss: 0.1899
Batch 320 - Loss: 0.5510
Batch 330 - Loss: 0.1699
Batch 340 - Loss: 0.2909
Batch 350 - Loss: 0.2806
Batch 360 - Loss: 0.1766
Batch 370 - Loss: 0.1701
Batch 380 - Loss: 0.1918
Batch 390 - Loss: 0.1341
Batch 400

In [14]:
!nvidia-smi

Tue May 19 11:11:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             34W /   70W |    4711MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
# 6. Simpan Model
model.save_pretrained("./donut-sroie-finetuned")
processor.save_pretrained("./donut-sroie-finetuned")
print("\nTraining selesai dan model disimpan!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training selesai dan model disimpan!


In [16]:
!zip -r donut-sroie-finetuned.zip ./donut-sroie-finetuned

files.download('donut-sroie-finetuned.zip')

  adding: donut-sroie-finetuned/ (stored 0%)
  adding: donut-sroie-finetuned/model.safetensors (deflated 7%)
  adding: donut-sroie-finetuned/generation_config.json (deflated 58%)
  adding: donut-sroie-finetuned/config.json (deflated 74%)
  adding: donut-sroie-finetuned/processor_config.json (deflated 55%)
  adding: donut-sroie-finetuned/tokenizer_config.json (deflated 48%)
  adding: donut-sroie-finetuned/tokenizer.json (deflated 74%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
model_path = "./donut-sroie-finetuned"
processor = DonutProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path)
model.to(device)
model.eval()

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

VisionEncoderDecoderModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0): DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )

In [18]:
# 2. Setup Dataloader untuk Test (Pastikan path mengarah ke folder test)
test_dataset = SROIEDataset(dataset_path="/content/SROIE2019/test", processor=processor)
test_dataloader = DataLoader(test_dataset, batch_size=1)

In [19]:
total_sim = 0.0

print("Mulai Evaluasi...\n")
with torch.no_grad(): # Matikan perhitungan gradien
    for idx, batch in enumerate(test_dataloader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        # 3. Generate teks dari gambar
        outputs = model.generate(
            pixel_values,
            max_length=512,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            decoder_start_token_id=model.config.decoder_start_token_id
        )

        # 4. Decode hasil prediksi
        pred_str = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        # 5. Decode label asli (ubah padding -100 jika ada ke pad_token_id)
        labels[labels == -100] = processor.tokenizer.pad_token_id
        true_str = processor.batch_decode(labels, skip_special_tokens=True)[0]

        # 6. Hitung rasio kemiripan (0.0 - 1.0)
        sim = difflib.SequenceMatcher(None, true_str, pred_str).ratio()
        total_sim += sim

        # Tampilkan 3 contoh pertama agar kebayang hasilnya
        if idx < 3:
            print(f"--- Gambar {idx+1} ---")
            print(f"Ground Truth : {true_str}")
            print(f"Prediksi     : {pred_str}")
            print(f"Kemiripan    : {sim:.2f}\n")

print(f"Evaluasi Selesai! Rata-rata Akurasi (String Similarity): {total_sim / len(test_dataloader):.2f}")

Mulai Evaluasi...

--- Gambar 1 ---
Ground Truth : <s_receipt> {"company": "OJC MARKETING SDN BHD", "date": "15/01/2019", "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B1750 MASAI, JOHOR", "total": "193.00"}</s_receipt>
Prediksi     : <s_receipt><s_receipt> {"company": "*** COPY ** O3C MARKETING SDN BHD", "date": "15/01/2019", "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B1750 MASAI, JOHOR", "total": "193.00"}</s_receipt>
Kemiripan    : 0.93

--- Gambar 2 ---
Ground Truth : <s_receipt> {"company": "OJC MARKETING SDN BHD", "date": "02/01/2019", "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR", "total": "170.00"}</s_receipt>
Prediksi     : <s_receipt><s_receipt> {"company": "OIMARKETING SDN BHD", "date": "02/01/2019", "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B1750 MASAI, JOHOR", "total": "170.00"}</s_receipt>
Kemiripan    : 0.95

--- Gambar 3 ---
Ground Truth : <s_receipt> {"company": "PERNIAGAAN ZHENG HUI", "date": "09/02/2018", "ad

In [ ]:
print("Silakan upload gambar struk:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

# 2. Load Model & Processor (Ganti path jika sudah di-download dan di-upload ke tempat lain)
model_path = "./donut-sroie-finetuned"
processor = DonutProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 3. Preprocessing Gambar & Setup Prompt
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
task_prompt = "<s_receipt>"
decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

# 4. Generate Teks
with torch.no_grad():
    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=512,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True
    )

# 5. Decode Hasil
hasil_teks = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print("\n--- Hasil Ekstraksi ---")
print(hasil_teks)

Silakan upload gambar struk:


Saving X51005255805.jpg to X51005255805.jpg


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]


--- Hasil Ekstraksi ---
<s_receipt><s_receipt> {"company": "SAM SAM TRADING CO", "date": "25/63", "address": "67,JLN MEWAH 25/63 TMN SRI MUDA, 40400 SHAH ALAM.", "total": "15.90"}</s_receipt>
